In [1]:
import sys

from src.data.load_cifar100 import get_cifar100_loaders, create_and_load_subset

sys.path.append('..')

import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.tensorboard import SummaryWriter
import numpy as np
import pandas as pd
import os
import datetime
from src.models.masked_autoencoder import MAE
from src.data.load_cifar10 import get_cifar10_loaders, create_and_load_subset_c10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cuda


In [2]:
# Dane
# train_loader, val_loader, test_loader = get_cifar10_loaders(batch_size=64)
# train_loader, val_loader, test_loader = get_cifar100_loaders(batch_size=64)
_, selected_classes, train_loader, val_loader, test_loader = create_and_load_subset(
    num_classes=2,
    batch_size=64,
    selected_classes = [60, 66],
)
print(f"Trenowanie na klasach: {selected_classes}")

Używam podanych klas: [60, 66]
Trenowanie na klasach: [60, 66]


In [3]:
# Model
model = MAE().to(device)
# criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [4]:
# Trening
BASE_DIR = os.getcwd()
# save_dir = os.path.join(BASE_DIR, '..', 'training_results', 'masked_autoencoder', 'cifar10')
save_dir = os.path.join(BASE_DIR, '..', 'training_results', 'masked_autoencoder', 'cifar100', f'{len(selected_classes)}_classes')
print(save_dir)
# writer_dir = os.path.join(BASE_DIR, '..', 'training_results', 'tb_mae', 'cifar10')
writer_dir = os.path.join(BASE_DIR, '..', 'training_results', 'tb_mae', 'cifar100', f'{len(selected_classes)}_classes')
os.makedirs(save_dir, exist_ok=True)
os.makedirs(writer_dir, exist_ok=True)
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
writer = SummaryWriter(os.path.join(writer_dir, f'mae_{timestamp}'))

num_epochs = 250

train_losses = []
val_losses = []
epoch_number = 0
best_val_loss = float('inf')
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0

    for batch_idx, (image, _) in enumerate(train_loader):
        image = image.to(device)

        reconstructed, x_masked, mask = model(image)
        loss = model.compute_loss(image, reconstructed, mask)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        if batch_idx % 100 == 0:
            print(f"  [{epoch+1}/{num_epochs}] Batch {batch_idx}/{len(train_loader)} "
                  f"Loss: {loss.item():.4f}")

    train_loss /= len(train_loader)
    train_losses.append(train_loss)
    writer.add_scalar('Loss/train', train_loss, epoch)

    # validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for image, _ in val_loader:
            image = image.to(device)

            reconstructed, x_masked, mask = model(image)
            loss = model.compute_loss(image, reconstructed, mask)
            val_loss += loss.item()

        if epoch % 5 == 0:
            n_images = min(8, image.size(0))

            # Oryginalne obrazy
            writer.add_images('Original', image[:n_images], epoch)

            # Zamaskowane obrazy
            writer.add_images('Masked', x_masked[:n_images], epoch)

            # Rekonstrukcje
            writer.add_images('Reconstructed', reconstructed[:n_images], epoch)

            # Wizualizacja maski (powtórzona dla 3 kanałów)
            mask_vis = mask[:n_images].repeat(1, 3, 1, 1)
            writer.add_images('Mask', mask_vis, epoch)
    val_loss /= len(val_loader)
    val_losses.append(val_loss)
    writer.add_scalar('Loss/val', val_loss, epoch)

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")


    # save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
            'latent_dim': 256,
            'train_losses': train_losses,
            'val_losses': val_losses,
            'selected_classes': selected_classes,
        }
        timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        # checkpoint_path = os.path.join(save_dir, f'masked_autoencoder_cifar10_best_{timestamp}.pt')
        checkpoint_path = os.path.join(save_dir, f'masked_autoencoder_cifar100_best_{timestamp}.pt')
        torch.save(checkpoint, checkpoint_path)


df = pd.DataFrame({
    'epoch': range(1, num_epochs + 1),
    'train_loss': train_losses,
    'val_loss': val_losses
})
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
# history_csv = os.path.join(save_dir, f'masked_autoencoder_cifar10_training_results_{timestamp}.csv')
history_csv = os.path.join(save_dir, f'masked_autoencoder_cifar100_training_results_{timestamp}_.csv')
df.to_csv(history_csv, index=False)


In [5]:
model_configs = [
    {'filename': 'masked_autoencoder_cifar100_best_20260114_002101.pt', 'model_id': 1},
    {'filename': 'masked_autoencoder_cifar100_best_20260114_002552.pt', 'model_id': 2},
    {'filename': 'masked_autoencoder_cifar100_best_20260114_003002.pt', 'model_id': 3},
    {'filename': 'masked_autoencoder_cifar100_best_20260114_003236.pt', 'model_id': 4},
    {'filename': 'masked_autoencoder_cifar100_best_20260114_003442.pt', 'model_id': 5},
    {'filename': 'masked_autoencoder_cifar100_best_20260114_003947.pt', 'model_id': 6},
    {'filename': 'masked_autoencoder_cifar100_best_20260114_154104.pt', 'model_id': 7},
    {'filename': 'masked_autoencoder_cifar100_best_20260114_155946.pt', 'model_id': 8},
]


save_dir = os.path.join(os.getcwd(), '..', 'training_results', 'masked_autoencoder', 'cifar100', '2_classes')


num_classes = 2
batch_size = 64


all_results = []


for config in model_configs:
    filename = config['filename']
    model_id = config['model_id']
    model_path = os.path.join(save_dir, filename)

    print(f"\n{'='*80}")
    print(f"Przetwarzanie modelu {model_id}/{len(model_configs)}: {filename}")
    print(f"{'='*80}\n")

    if not os.path.exists(model_path):
        print(f"UWAGA: Plik {filename} nie istnieje.")
        continue

    best_checkpoint = torch.load(model_path, map_location=device)
    saved_classes = best_checkpoint['selected_classes']


    _, _, _, _, test_loader = create_and_load_subset(
        selected_classes=saved_classes,
        num_classes=num_classes,
        batch_size=batch_size
    )


    model = MAE().to(device)
    model.load_state_dict(best_checkpoint['model_state_dict'])
    model.eval()

    test_loss = 0
    num_batches = 0

    with torch.no_grad():
        for images, _ in test_loader:
            images= images.to(device)
            reconstructed, x_masked, mask = model(images)
            loss = model.compute_loss(images, reconstructed, mask)
            test_loss += loss.item()
            num_batches += 1

    test_loss /= num_batches

    print(f"Test Loss: {test_loss:.6f}")


    all_results.append({
        'model_id': model_id,
        'model_filename': filename,
        'test_loss': test_loss,
        'num_classes': num_classes
    })

    print(f"\nZakończono przetwarzanie modelu {model_id}")


print("Zapisywanie wyników do pliku CSV...")
results_df = pd.DataFrame(all_results)
csv_path = os.path.join(save_dir, f'all_models_test_loss_{num_classes}_classes.csv')
results_df.to_csv(csv_path, index=False)
print(f"Zapisano wyniki do: {csv_path}")


print(f"PODSUMOWANIE WSZYSTKICH {len(all_results)} MODELI ({num_classes} KLAS)")

print(results_df.to_string())


print("\n" + "="*80)
print("STATYSTYKI TEST LOSS")
print("="*80)
mean_loss = results_df['test_loss'].mean()
std_loss = results_df['test_loss'].std()
min_loss = results_df['test_loss'].min()
max_loss = results_df['test_loss'].max()

print(f"Średnia:        {mean_loss:.6f}")
print(f"Odchylenie std: {std_loss:.6f}")
print(f"Minimum:        {min_loss:.6f}")
print(f"Maximum:        {max_loss:.6f}")
print("="*80)



# save_dir = os.path.join(os.getcwd(), '..', 'training_results', 'masked_autoencoder', 'cifar100', '50_classes')
# best_checkpoint = torch.load(os.path.join(save_dir, 'masked_autoencoder_cifar100_best_20260114_013421.pt'))
#
# saved_classes = best_checkpoint['selected_classes']
# _, _, _, _, test_loader = create_and_load_subset(
#     selected_classes=saved_classes,
#     num_classes=50,
#     batch_size=64
# )
# model.load_state_dict(best_checkpoint['model_state_dict'])
# correct_test = 0
# total_test = 0
# test_loss = 0
#
# model.eval()
# print("\nCalculating metrics on test set...")
# with torch.no_grad():
#     for image, _ in test_loader:
#         image = image.to(device)
#
#         reconstructed, x_masked, mask = model(image)
#         loss = model.compute_loss(image, reconstructed, mask)
#         test_loss += loss.item()
#
# test_loss /= len(test_loader)
# # writer.add_scalar('Loss/test', test_loss)
# print("FINAL EVALUATION RESULTS")
# print(f"Test Loss:           {test_loss:.6f}")
#
    


Przetwarzanie modelu 1/8: masked_autoencoder_cifar100_best_20260114_002101.pt

Używam podanych klas: [14, 58]
Test Loss: 0.092259

Zakończono przetwarzanie modelu 1

Przetwarzanie modelu 2/8: masked_autoencoder_cifar100_best_20260114_002552.pt

Używam podanych klas: [58, 85]
Test Loss: 0.092649

Zakończono przetwarzanie modelu 2

Przetwarzanie modelu 3/8: masked_autoencoder_cifar100_best_20260114_003002.pt

Używam podanych klas: [11, 1]
Test Loss: 0.083613

Zakończono przetwarzanie modelu 3

Przetwarzanie modelu 4/8: masked_autoencoder_cifar100_best_20260114_003236.pt

Używam podanych klas: [54, 13]
Test Loss: 0.097891

Zakończono przetwarzanie modelu 4

Przetwarzanie modelu 5/8: masked_autoencoder_cifar100_best_20260114_003442.pt

Używam podanych klas: [9, 35]
Test Loss: 0.083509

Zakończono przetwarzanie modelu 5

Przetwarzanie modelu 6/8: masked_autoencoder_cifar100_best_20260114_003947.pt

Używam podanych klas: [17, 4]
Test Loss: 0.065822

Zakończono przetwarzanie modelu 6

Przetw